In [ ]:
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from PyPDF2 import PdfReader
import re

# Load environment variables
load_dotenv()

In [ ]:
model = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("BASE_URL"),
    openai_api_key=os.getenv("OPENAI_API_KEY")
)


In [ ]:
from crewai.tools import tool

@tool("fetch_pdf_content")
def fetch_pdf_content(pdf_path: str) -> str:
    """Reads a local PDF and returns the content.

    Args:
        pdf_path: Path to the PDF file to read

    Returns:
        Extracted text content from the PDF
    """
    try:
        with open(pdf_path, 'rb') as f:
            pdf = PdfReader(f)
            text = '\n'.join(page.extract_text() for page in pdf.pages if page.extract_text())

        processed_text = re.sub(r'\s+', ' ', text).strip()
        return processed_text
    except Exception as e:
        return f"Error reading PDF: {str(e)}"


In [ ]:
pdf_reader = Agent(
    role='PDF Content Extractor',
    goal='Extract and preprocess text from a PDF located in current local directory',
    backstory='Specializes in handling and interpreting PDF documents',
    verbose=True,
    tools=[fetch_pdf_content],
    allow_delegation=False,
    llm=model
)

article_writer = Agent(
    role='Article Creator',
    goal='Write a concise and engaging article',
    backstory='Expert in creating informative and engaging articles',
    verbose=True,
    allow_delegation=False,
    llm=model
)

title_creator = Agent(
    role='Title Generator',
    goal='Generate a compelling title for the article',
    backstory='Skilled in crafting engaging and relevant titles',
    verbose=True,
    allow_delegation=False,
    llm=model
)

In [ ]:
def pdf_reading_task(pdf_path):
    return Task(
        description=f"Read the PDF file at {pdf_path} and extract all text content. Process the text by removing extra whitespace and return the cleaned content.",
        agent=pdf_reader,
        expected_output="Extracted and preprocessed text from the PDF",
    )


task_article_drafting = Task(
    description="Create a concise article with 8-10 paragraphs based on the extracted PDF content.",
    agent=article_writer,
    expected_output="8-10 paragraphs describing the key points of the PDF",
)

task_title_generation = Task(
    description="Generate an engaging and relevant title for the article.",
    agent=title_creator,
    expected_output="A Title of About 5-7 Words"
)

In [ ]:
# Define the PDF path from environment variable
pdf_local_relative_path = os.getenv("PDF_PATH")

crew = Crew(
    agents=[pdf_reader, article_writer, title_creator],
    tasks=[pdf_reading_task(pdf_local_relative_path),
           task_article_drafting,
           task_title_generation],
    verbose=True  # Changed from verbose=2 to verbose=True
)

# Let's start!
result = crew.kickoff()